# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a comprehensive guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata attributes safely
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Dataset identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Date published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² dataset contains record sets (datasets or tables), each uniquely identified by an `@id`.
Let's enumerate the record sets, their fields, and columns by their `@id`.

In [ ]:
# List all record sets and their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    Field @id: {f['@id']}, Name: {f.get('name','')}, DataType: {f.get('dataType','')}")
                columns = f.get('column', [])
                if columns:
                    print("    Columns:")
                    for c in columns:
                        print(f"      Column @id: {c['@id']} Name: {c.get('name','')}")
        else:
            print("  No fields found.")
        print()

# For demonstration (and subsequent steps), get the first record set @id
if record_sets:
    primary_record_set_id = record_sets[0]['@id']
    print(f"Primary record set selected: {primary_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we extract the data from the primary record set using its `@id` and examine the available columns.

In [ ]:
# Extract data from the primary record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet] if dataset.metadata.recordSet else []

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the primary record set
if record_set_ids:
    df = dataframes[record_set_ids[0]]
    print(f"Columns in record set {record_set_ids[0]}:")
    print(df.columns.tolist())
    print(df.head())
else:
    print("No record sets found to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by numeric fields, normalizing, and grouping.

We'll select a numeric field (e.g., the total score on the PHQ-9 depression scale) using its `@id` and perform some processing.

In [ ]:
# Find a numeric field for EDA
numeric_field_id = None
group_field_id = None
if dataset.metadata.recordSet:
    primary_rs = dataset.metadata.recordSet[0]
    flds = primary_rs.get('field', [])
    for f in flds:
        # Example: PHQ-9 score field (assume dataType Integer or Float and 'phq9_score' in name)
        if f.get('dataType') in ['schema:Integer', 'schema:Float']:
            if 'phq' in f.get('name','').lower():
                numeric_field_id = f['@id']
            elif not group_field_id and 'education' in f.get('name','').lower():
                group_field_id = f['@id']

if not numeric_field_id and df is not None:
    # Fallback: use first numeric column in DataFrame
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

# Use a threshold and filter the records
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. For example, visualize the PHQ-9 score distribution or group by level of education (all referenced by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for numeric field
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped boxplot if group field is available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped: no numeric field found.")

## 6. Conclusion
In this notebook, we explored a FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. 

- We loaded metadata and inspected the schema using Croissant entity `@id`s.
- Extracted records and reviewed the structure of fields and columns.
- Performed basic EDA, including filtering and normalization using field `@id`s, and grouped data by demographic attribute.
- Visualized the distribution and grouping using the provided schema mappings.

This approach demonstrates reproducible, transparent data handling with explicit reference to semantic identifiers, enhancing interoperability and FAIRness in AI-ready datasets.